### N-граммные языковые модели, или как писать научные статьи (4 балла)

Обучим языковую модель на корпусе статей с [ArXiv](http://arxiv.org/) и попробуем сгенерировать собственную научную публикацию!

![img](../resources/monkey.jpg)

_Данные: neelshah18, [Kaggle](https://www.kaggle.com/neelshah18/arxivdataset/)._

_Предупреждение: к настоящей науке это отношения не имеет. Но эксперимент забавный!_

> Неофициальный перевод [оригинального семинара](./seminar.ipynb). Код и проверки сохранены.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
# Альтернативная ссылка для ручного скачивания: https://yadi.sk/d/_nGyU2IajjR9-w
!wget "https://www.dropbox.com/s/99az9n1b57qkd9j/arxivData.json.tar.gz?dl=1" -O arxivData.json.tar.gz
!tar -xvzf arxivData.json.tar.gz
data = pd.read_json("./arxivData.json")
data.sample(n=5)

In [ ]:
# собери строки: объедини заголовок и описание
lines = data.apply(lambda row: row['title'] + ' ; ' + row['summary'].replace("\n", ' '), axis=1).tolist()

sorted(lines, key=len)[:3]

### Токенизация

Всё как обычно: данные неочищенные. Приведи текст в порядок. Можно использовать `WordPunctTokenizer` или другой подходящий токенизатор.

In [ ]:
# Задание: преобразуй строки на месте в токены, разделённые пробелами. Используй WordPunctTokenizer.

<YOUR CODE>
lines = <...>

In [ ]:
assert sorted(lines, key=len)[0] == \
    'differential contrastive divergence ; this paper has been retracted .'
assert sorted(lines, key=len)[2] == \
    'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .'

### N-граммная языковая модель (1 балл)

Языковая модель оценивает вероятность текста — совместную вероятность всех его токенов $w_t$:
$P(X) = P(w_1, \dots, w_T)$.

По правилу цепочки:

$$
P(w_1, \dots, w_T) = P(w_1)P(w_2 \mid w_1)\dots P(w_T \mid w_1, \dots, w_{T-1})
$$

Проблема в том, что последний множитель $P(w_T \mid w_1, \dots, w_{T-1})$ зависит от всех предыдущих слов. Для длинного текста, например при $T=1000$, оценить такую вероятность практически невозможно.

Популярное приближение: следующее слово зависит лишь от ограниченного числа предыдущих слов:

$$
P(w_t \mid w_1, \dots, w_{t - 1}) = P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1})
$$

Такую модель называют **n-граммной языковой моделью**, где $n$ — гиперпараметр. Например, в триграммной модели каждое слово зависит только от двух предыдущих.

$$
P(w_1, \dots, w_n) = \prod_t P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1}).
$$

Это приближение также известно как **марковское предположение порядка $n$**.

Сначала нужно подсчитать, сколько раз каждое слово встречается после заданной последовательности из $N-1$ предыдущих слов.

In [ ]:
from tqdm import tqdm
from collections import defaultdict, Counter

# специальные токены:
# - `UNK` represents absent tokens,
# - `EOS` is a special token after the end of sequence

UNK, EOS = "_UNK_", "_EOS_"

def count_ngrams(lines, n):
    """
    Count how many times each word occured after (n - 1) previous words
    :param lines: an iterable of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    When building counts, please consider the following two edge cases:
    - every prefix must contain exactly (n - 1) tokens. If a prefix is shorter than that,
      pad it with UNK on the left. The number of padding tokens therefore depends on n:
      derive it from n, do not hard-code it. For n=3,
      empty prefix: "" -> (UNK, UNK)
      short prefix: "the" -> (UNK, the)
      long prefix: "the new approach" -> (new, approach)
      For n=2 the very same prefixes give (UNK,), (the,) and (approach,), and for n=1 all
      of them give the empty tuple (). Equivalently: prepend (n - 1) UNK tokens to a line,
      then slide a window of size n over it.
    - you should add a special token, EOS, at the end of each sequence
      "... with deep neural networks ." -> (..., with, deep, neural, networks, ., EOS)
      count the probability of this token just like all others.

    Useful invariant: a line with L tokens always contributes exactly L + 1 counted events
    (its L tokens plus EOS), for every n. Note also that NGramLanguageModel below already
    implements this padding rule in get_possible_next_tokens - keep the two consistent, or
    the model will look up prefixes that you never counted.
    """
    counts = defaultdict(Counter)
    # counts[(word1, word2)][word3] — сколько раз word3 встретилось после (word1, word2)

    <YOUR CODE>

    return counts


In [ ]:
# проверим результат
dummy_lines = sorted(lines, key=len)[:100]
dummy_counts = count_ngrams(dummy_lines, n=3)
assert set(map(len, dummy_counts.keys())) == {2}, "please only count {n-1}-grams"
assert len(dummy_counts[('_UNK_', '_UNK_')]) == 78
assert dummy_counts['_UNK_', 'a']['note'] == 3
assert dummy_counts['p', '=']['np'] == 2
assert dummy_counts['author', '.']['_EOS_'] == 1

# правило padding должно выполняться для любого n, а не только n=3
for n in (1, 2, 3, 5, 10):
    counts_n = count_ngrams(dummy_lines, n=n)
    assert set(map(len, counts_n.keys())) == {n - 1}, \
        "every prefix must have exactly (n - 1) tokens, got lengths %s for n=%i" % (
            set(map(len, counts_n.keys())), n)
    assert sum(sum(c.values()) for c in counts_n.values()) == 2877, \
        "each line must contribute exactly len(tokens) + 1 events (its tokens plus EOS) " \
        "for every n. If this passes for n=3 but fails otherwise, you have most likely " \
        "hard-coded the number of UNK padding tokens instead of deriving it from n."

Когда мы умеем считать N-граммы, можно построить вероятностную языковую модель.

Простейшая оценка вероятности пропорциональна частоте соответствующей N-граммы:

$$ P(w_t | prefix) = { Count(prefix, w_t) \over \sum_{\hat w} Count(prefix, \hat w) } $$

In [ ]:
class NGramLanguageModel:
    def __init__(self, lines, n):
        """
        Train a simple count-based language model:
        compute probabilities P(w_t | prefix) given ngram counts

        :param n: computes probability of next token given (n - 1) previous words
        :param lines: an iterable of strings with space-separated tokens
        """
        assert n >= 1
        self.n = n

        counts = count_ngrams(lines, self.n)

        # вычисли вероятности токенов по счётчикам
        self.probs = defaultdict(Counter)
        # probs[(word1, word2)][word3] = P(word3 | word1, word2)

        # заполни self.probs фактическими вероятностями
        <YOUR CODE>

    def get_possible_next_tokens(self, prefix):
        """
        :param prefix: string with space-separated prefix tokens
        :returns: a dictionary {token : it's probability} for all tokens with positive probabilities
        """
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.n + 1):]
        prefix = [ UNK ] * (self.n - 1 - len(prefix)) + prefix
        return self.probs[tuple(prefix)]

    def get_next_token_prob(self, prefix, next_token):
        """
        :param prefix: string with space-separated prefix tokens
        :param next_token: the next token to predict probability for
        :returns: P(next_token|prefix) a single number, 0 <= P <= 1
        """
        return self.get_possible_next_tokens(prefix).get(next_token, 0)

Проверим модель!

In [ ]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

p_initial = dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']
assert np.allclose(p_initial['learning'], 0.02)
assert np.allclose(p_initial['a'], 0.13)
assert np.allclose(p_initial.get('meow', 0), 0)
assert np.allclose(sum(p_initial.values()), 1)

p_a = dummy_lm.get_possible_next_tokens('a') # '' -> ['_UNK_', 'a']
assert np.allclose(p_a['machine'], 0.15384615)
assert np.allclose(p_a['note'], 0.23076923)
assert np.allclose(p_a.get('the', 0), 0)
assert np.allclose(sum(p_a.values()), 1)

assert np.allclose(dummy_lm.get_possible_next_tokens('a note')['on'], 1)
assert dummy_lm.get_possible_next_tokens('a machine') == \
    dummy_lm.get_possible_next_tokens("there have always been ghosts in a machine"), \
    "your 3-gram model should only depend on 2 previous words"

Теперь, когда N-граммная модель работает, посмотрим, какие последовательности она может генерировать. Но сначала обучим её на всём датасете.

In [ ]:
lm = NGramLanguageModel(lines, n=3)

Генерация происходит **последовательно**: поддерживаем список уже полученных токенов и на каждом шаге случайно выбираем следующий токен в соответствии с вероятностями модели.

$ X = [] $

**Повторять бесконечно:**
- $w_{next} \sim P(w_{next} | X)$
- $X = concat(X, w_{next})$

Вместо обычного вероятностного выбора можно всегда брать самый вероятный токен, выбирать среди top-K кандидатов либо использовать **температуру**. В последнем случае выборка делается из распределения:

$$w_{next} \sim {P(w_{next} | X) ^ {1 / \tau} \over \sum_{\hat w} P(\hat w | X) ^ {1 / \tau}}$$

Где $\tau > 0$ — температура модели. При $\tau \ll 1$ наиболее вероятные токены выбираются ещё чаще, а маловероятные почти исчезают.

In [ ]:
def get_next_token(lm, prefix, temperature=1.0):
    """
    return next token after prefix;
    :param temperature: samples proportionally to lm probabilities ^ (1 / temperature)
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """
    <YOUR CODE>

In [ ]:
from collections import Counter
test_freqs = Counter([get_next_token(lm, 'there have') for _ in range(10000)])
assert 250 < test_freqs['not'] < 450
assert 8500 < test_freqs['been'] < 9500
assert 1 < test_freqs['lately'] < 200

test_freqs = Counter([get_next_token(lm, 'deep', temperature=1.0) for _ in range(10000)])
assert 1500 < test_freqs['learning'] < 3000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.5) for _ in range(10000)])
assert 8000 < test_freqs['learning'] < 9000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.0) for _ in range(10000)])
assert test_freqs['learning'] == 10000

print("Looks nice!")

Поэкспериментируем с моделью.

In [ ]:
prefix = 'artificial' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

In [ ]:
prefix = 'bridging the' # <- more of your ideas

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break

print(prefix)

**Продолжение в домашнем задании:** nucleus sampling, top-k sampling и beam search (для любителей сложных задач).

### Оценка языковых моделей: перплексия (1 балл)

**Перплексия (perplexity)** измеряет, насколько хорошо модель приближает реальное распределение вероятностей данных. **Чем меньше перплексия, тем лучше модель.**

Для одного предложения:

$$
{\mathbb{P}}(w_1 \dots w_N) = P(w_1, \dots, w_N)^{-\frac1N} = \left( \prod_t P(w_t \mid w_{t - n}, \dots, w_{t - 1})\right)^{-\frac1N},
$$

На уровне корпуса перплексия — произведение вероятностей всех токенов во всех предложениях в степени $-\frac1N$, где $N$ — **общее число оцениваемых токенов**.

Предложение $w_1\dots w_L$ даёт $L+1$ множитель, поскольку EOS предсказывается и оценивается как обычный токен:

$$P(w_1 \mid \varnothing) \cdot P(w_2 \mid w_1) \cdots P(w_L \mid w_1 \dots w_{L-1}) \cdot P(EOS \mid w_1 \dots w_L)$$

Здесь $\varnothing$ — пустой префикс. EOS также учитывается в $N$, поэтому $N = \sum_{\text{sentences}} (L + 1)$ — то же число событий, которое считалось в \`count_ngrams\`.

Произведение вероятностей может стать слишком маленьким для float32/float64. Поэтому рекомендуем сначала вычислить логарифм перплексии через логарифмы вероятностей, а затем взять экспоненту.

In [ ]:
def perplexity(lm, lines, min_logprob=np.log(10 ** -50.)):
    """
    :param lines: a list of strings with space-separated tokens
    :param min_logprob: if log(P(w | ...)) is smaller than min_logprop, set it equal to min_logrob
    :returns: corpora-level perplexity - a single scalar number from the formula above

    Note: scoring a line "w_1 w_2 ... w_L" means scoring L + 1 events:
        P(w_1 | ""), P(w_2 | "w_1"), ..., P(w_L | "w_1 ... w_{L-1}"), P(EOS | "w_1 ... w_L")
    The prefix of the very first token is the empty string "", i.e. an empty prefix and not
    a string of UNKs: the model pads it to (n - 1) UNK tokens on its own inside
    get_possible_next_tokens, so do not add any padding here. Likewise, w_1 is simply the
    first real token of the line. The trailing EOS is scored like any other token and is
    counted in N too, so a line with L tokens adds L + 1 to N.

    PLEASE USE lm.get_next_token_prob and NOT lm.get_possible_next_tokens
    """
    <YOUR CODE>

    return <...>

In [ ]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)
ppx_missing = perplexity(lm3, ['the jabberwock , with eyes of flame , '])  # thanks, L. Carrol

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f" % (ppx1, ppx3, ppx10))

assert all(0 < ppx < 500 for ppx in (ppx1, ppx3, ppx10)), "perplexity should be non-negative and reasonably small"
assert ppx1 > ppx3 > ppx10, "higher N models should overfit and "
assert np.isfinite(ppx_missing) and ppx_missing > 10 ** 6, "missing words should have large but finite perplexity. " \
    " Make sure you use min_logprob right"
assert np.allclose([ppx1, ppx3, ppx10], (318.2132342216302, 1.5199996213739575, 1.1838145037901249))

Теперь измерим перплексию на практике: разделим данные на обучающую и тестовую выборки и оценим модель только на тестовых данных.

In [ ]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(lines, test_size=0.25, random_state=42)

for n in (1, 2, 3):
    lm = NGramLanguageModel(n=n, lines=train_lines)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))


In [ ]:
# упс, значение выросло до бесконечности :)

### Сглаживание языковой модели

Проблема нашей простой модели: если встретилась неизвестная N-грамма, она получает вероятность 0. В результате перплексия становится бесконечной.

Чтобы этого избежать, используют **сглаживание**: корректируют частоты так, чтобы вероятности не становились слишком маленькими.

Самый простой метод — аддитивное сглаживание (также известное как [сглаживание Лапласа](https://en.wikipedia.org/wiki/Additive_smoothing)):

$$ P(w_t | prefix) = { Count(prefix, w_t) + \delta \over \sum_{\hat w} (Count(prefix, \hat w) + \delta) } $$

Если после данного префикса встретилось мало слов, сглаживание делает распределение более равномерным. Обрати внимание: сумма в знаменателе берётся **по всем словам словаря**.

Ниже приведён готовый пример реализации.

In [ ]:
class LaplaceLanguageModel(NGramLanguageModel):
    """ this code is an example, no need to change anything """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.probs = defaultdict(Counter)

        for prefix in counts:
            token_counts = counts[prefix]
            total_count = sum(token_counts.values()) + delta * len(self.vocab)
            self.probs[prefix] = {token: (token_counts[token] + delta) / total_count
                                          for token in token_counts}
    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)
        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob = missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        return {token: token_probs.get(token, missing_prob) for token in self.vocab}

    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)
        if next_token in token_probs:
            return token_probs[next_token]
        else:
            missing_prob_total = 1.0 - sum(token_probs.values())
            missing_prob_total = max(0, missing_prob_total) # prevent rounding errors
            return missing_prob_total / max(1, len(self.vocab) - len(token_probs))


**Ограничение примера:** приведённая реализация считает одинаково вероятными все слова, неизвестные в данном контексте, **включая слова вне словаря**. Поэтому при встрече с такими словами перплексия получится искусственно заниженной, и сравнение с моделью, у которой меньше неизвестных слов, будет некорректным.

В собственной реализации можно добавить фиктивный токен \`UNK\` с ненулевой вероятностью. Формально это может привести к тому, что сумма вероятностей окажется не равна 1, но для учебного упражнения этого достаточно.

In [ ]:
# проверка корректности вероятностной модели
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [ ]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=0.1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

In [ ]:
# необязательно: попробуй генерировать токены из этой модели

### Сглаживание Кнезера — Нея (2 балла)

Аддитивное сглаживание просто и достаточно полезно, но это не самый современный алгоритм.

**Последнее задание в этом ноутбуке** — реализовать [сглаживание Кнезера — Нея](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing).

Для $n>1$ оно задаётся рекурсивно:

$$P_{kn}(w_t | prefix_{n-1}) = { \max(0, Count(prefix_{n-1}, w_t) - \delta) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)} + \lambda_{prefix_{n-1}} \cdot P_{kn}(w_t | prefix_{n-2})$$

Где:

- $prefix_{n-1}$ — кортеж из $n-1$ предыдущих токенов.
- $\lambda_{prefix_{n-1}}$ — нормировочная константа, выбранная так, чтобы сумма вероятностей была равна 1. Из каждого из $N_{1+}(prefix_{n-1}, \bullet)$ **различных** токенов, встречавшихся после данного префикса, вычитается $\delta$. Высвободившаяся масса вероятности перераспределяется:

$$\lambda_{prefix_{n-1}} = {\delta \cdot N_{1+}(prefix_{n-1}, \bullet) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)}$$

- $P_{kn}(w_t | prefix_{n-2})$ — модель более низкого порядка (сглаживание Кнезера — Нея для $(n-1)$-грамм, то есть та же формула без самого старого токена префикса).
- Униграммная вероятность $P_{kn}(w_t)$ — специальный базовый случай, завершающий рекурсию: вероятность увидеть $w_t$ в незнакомом контексте.

**Префиксы, которых не было при обучении.** Если $Count(prefix_{n-1}, \hat w)=0$ для всех $\hat w$, знаменатель равен нулю, и первые два слагаемых дают неопределённость $0/0$. Данных высокого порядка нет: нужно полностью перейти к модели низшего порядка, приняв $\lambda_{prefix_{n-1}}=1$ и первое слагаемое равным нулю. Возвращай $P_{kn}(w_t | prefix_{n-2})$ без изменений. Полученное распределение остаётся корректным; на тестовой выборке такие случаи будут встречаться часто.

**Выбор $\delta$.** Используй $\delta \in (0,1]$. Частоты — целые числа $\ge 1$, поэтому при таком ограничении $\max(0, Count-\delta)=Count-\delta$, а вероятности в сумме точно дают 1. При $\delta>1$ отсечение через $\max$ приводит к несовпадению вычтенной и перераспределённой масс: тест нормировки не пройдёт.

Подробные формулы смотри в лекции или Википедии.

**Твоя задача:**
1. Реализовать класс \`KneserNeyLanguageModel\`.
2. Проверить его на униграммной, биграммной и триграммной моделях.
3. Подобрать разумное оптимальное значение $\delta$ для триграммной модели со сглаживанием Кнезера — Нея.

_Примечание: тест перебирает \`dummy_lm.vocab\`, поэтому класс должен предоставлять \`self.vocab\` — множество всех известных токенов, как в \`LaplaceLanguageModel\`._

In [ ]:
class KneserNeyLanguageModel(NGramLanguageModel):
    """ A template for Kneser-Ney language model. Default delta may be suboptimal. """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        # подсказка: потребуются частоты n-грамм всех порядков от 1 до n и множество self.vocab
        <YOUR CODE>

    def get_possible_next_tokens(self, prefix):
        < YOUR CODE >

    def get_next_token_prob(self, prefix, next_token):
        # не забудь обработать префикс, отсутствовавший в обучении (см. выше)
        <YOUR CODE>

In [ ]:
# проверка корректности вероятностной модели
for n in (1, 2, 3):
    dummy_lm = KneserNeyLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [ ]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, delta=<...>)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))